# RGB + depth face recognition: embedding fusion

This notebook recognises people from paired **colour images** and **depth point clouds**:

* `Raw_{image_id}_{subject_id}_{timestamp}_color.png`
* `Raw_{image_id}_{subject_id}_{timestamp}_points.npy`

Two frozen, pre-trained models produce one 512-d embedding per modality:
* **colour**: FaceNet (Inception-ResNet-v1 trained on VGGFace2), applied after MTCNN face detection and alignment;
* **depth**: your PK-trained point-cloud network from `face_pipeline.ipynb`.

A small **fusion network** then learns to combine them into one 256-d identity embedding:

```
colour  -> FaceNet  -> 512 -> [LayerNorm, FC, GELU, Dropout, FC] -> z_rgb   (256) --+
                                                                                  +--> [FC, GELU, Dropout, FC] -> fused (256, L2-normalised)
depth   -> PK model -> 512 -> [LayerNorm, FC, GELU, Dropout, FC] -> z_depth (256) --+
```

**Keeping the network from being "lazy".** A fusion network tends to lean on whichever modality is easiest and ignore the other. Here, that is usually colour. Three counter-measures:
1. **Modality dropout**: for a random 25% of training samples, one modality is replaced by a learned "missing" vector. The network can't count on either input always being there, so it must learn to use both.
2. **Auxiliary per-modality losses**: `z_rgb` and `z_depth` each get their own identity classifier, so each branch must stay discriminative on its own.
3. **Gradient modulation** (OGM, Peng et al., CVPR 2022): every step, the branch whose auxiliary classifier is currently more confident gets its gradients scaled down, so the weaker modality catches up.

The evaluation also measures **modality reliance**: how much the score drops when either input is hidden. A lazy model loses almost nothing when its ignored modality is hidden.

**Clean protocol.** Identities are split into **train / val / test with no person in two splits**. Val picks the epoch and the score-fusion weight. Test is used once, at the very end. Within val and test, each person's **earliest** scan is the gallery and the later scans are probes, as in real use: enrolment first, recognition later.

| # | Stage | Output (in `WORK_DIR`) | Skipped when |
|---|---|---|---|
| 1 | Index the dataset and inspect one raw scan | – | always runs (fast) |
| 2 | Detect faces in the colour images (MTCNN) | `detections.json` | the file exists |
| 3 | Preprocess raw point clouds to `[N, 7]` and audit their quality | `clouds/`, `cloud_failures.json`, `cloud_audit.json` | per scan, if its file exists |
| 4 | Identity-disjoint train / val / test split | `splits.json` | the file exists |
| 5 | Frozen embeddings, with augmented copies for training | `embeddings.npz` | the file exists |
| 6 | Baselines: each modality alone, and score-level fusion | – | always runs (fast) |
| 7 | Train the fusion network (early stopping on val) | `fusion/` | `fusion/final.pth.tar` exists |
| 8 | Final test evaluation and visualisations | `results_test.csv` | always runs |

Reusable pieces live in the repository (`pointnet2/data/raw_scan.py`, `pointnet2/utils/face_eval.py`). The stage logic and the training loop are in this notebook.

## 0. Setup

In [ ]:
import os, sys, glob, subprocess

REPO_URL    = "https://github.com/AdarWa/3DFacePointCloudNet.git"
REPO_BRANCH = "claude/busy-ritchie-gyjpka"
REPO_DIR    = "/kaggle/working/3DFacePointCloudNet"

for candidate in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.isdir(os.path.join(candidate, "pointnet2")):
        REPO_DIR = candidate
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Repository:", REPO_DIR)

In [ ]:
!pip install -q -r requirements.txt plotly pandas
!pip install -q --no-deps facenet-pytorch   # --no-deps: its pinned torch version would downgrade yours
if glob.glob("pointnet2/_ext*.so"):
    print("CUDA ops already built")
else:
    !python setup.py build_ext --inplace

In [ ]:
import os, json, zlib
from collections import defaultdict
from multiprocessing import Pool
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image
from sklearn.manifold import TSNE
from facenet_pytorch import MTCNN, InceptionResnetV1

# Repository utilities
from pointnet2.models import Pointnet2FaceClsSSG
from pointnet2.train.layer import MarginCosineProduct
from pointnet2.data import data_utils
from pointnet2.data.raw_scan import find_raw_pairs, load_raw_points, raw_to_cloud
from pointnet2.data.quality import audit_scans, flag_scans
from pointnet2.data.TripletNpyFaceDataset import find_npy_files, load_npy_cloud
from pointnet2.utils.face_eval import verification_metrics, similarity_metrics

# Silence "AssertionError: can only test a child process" from DataLoader clean-up in forked workers.
from torch.utils.data import dataloader as _dataloader
_MAIN_PID = os.getpid()
_iterator_del = _dataloader._MultiProcessingDataLoaderIter.__del__
def _iterator_del_in_owner_only(self):
    if os.getpid() == _MAIN_PID:
        _iterator_del(self)
_dataloader._MultiProcessingDataLoaderIter.__del__ = _iterator_del_in_owner_only

assert torch.cuda.is_available(), "This notebook needs a CUDA GPU (Kaggle: Settings > Accelerator > GPU)"
DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

## 1. Configuration

In [ ]:
# ---- paths ------------------------------------------------------------------
RAW_DATA       = "/kaggle/input/<your-dataset>"    # folder with the Raw_*_color.png / Raw_*_points.npy files
PK_CKPT        = "runs/pipeline/pk/best.pth.tar"   # depth model from face_pipeline.ipynb
REFERENCE_DATA = None    # optional: processed .npy scans the depth model was fine-tuned on (for a format check)
WORK_DIR       = "runs/fusion"

# ---- data -------------------------------------------------------------------
NUM_POINTS      = 2048   # must match what the depth model was trained with
CURVATURE_POWER = 0.1
NUM_WORKERS     = 4
SEED            = 0
EVAL_BATCH      = 64

SPLIT = dict(val=0.15, test=0.15,   # fractions of IDENTITIES (no person is in two splits)
             min_samples=2)         # identities need >= 2 usable samples (gallery + probe)

PREPROCESS = dict(camera="opencv",        # raw point axes: "opencv" = x right, y down, z forward (RealSense,
                                          # Kinect); "opengl" = y up, z towards the viewer
                  face_radius_mm=95.0,    # keep points within this distance of the nose tip
                  curvature_k=20,         # neighbours for normals / curvature
                  use_face_box=True,      # organized clouds: crop with the colour face box first
                  min_face_prob=0.95)     # MTCNN confidence needed to keep a sample

# ---- embeddings -------------------------------------------------------------
TRAIN_VIEWS = 6          # per training sample and modality: 1 clean + 5 augmented embeddings
RGB_AUG   = dict(max_rotation_deg=8.0, max_shift=0.05, scale=(0.95, 1.05),
                 brightness=0.25, contrast=0.25, erase_prob=0.3)
DEPTH_AUG = dict(max_rotation_deg=10.0, jitter_std=0.002, cutout_prob=0.5, cutout_max_frac=0.15)

# ---- fusion network ---------------------------------------------------------
FUSION = dict(proj_dim=256, hidden=512, dropout=0.3,
              epochs=200, batch_size=128, lr=1e-3, weight_decay=1e-4,
              patience=25,                    # early stopping: epochs without a better val score
              cosface_s=30.0, cosface_m=0.35)  # identity loss (CosFace) scale and margin
ANTI_LAZY = dict(modality_dropout=0.25,  # share of training samples with one modality hidden
                 aux_weight=0.5,         # weight of the per-modality auxiliary losses
                 ogm_alpha=0.5)          # gradient-modulation strength (0 = off)

SELECT_METRIC = "tar@far=0.01"   # val metric for early stopping and the score-fusion weight (higher is better)

## 2. Helpers

In [ ]:
os.makedirs(WORK_DIR, exist_ok=True)
def work(*parts):
    return os.path.join(WORK_DIR, *parts)

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=1)

def load_json(path):
    with open(path) as f:
        return json.load(f)

class Mean:
    """Running average."""
    def __init__(self):
        self.total, self.count = 0.0, 0
    def add(self, value, n=1):
        self.total += value * n
        self.count += n
    @property
    def value(self):
        return self.total / max(self.count, 1)

FRONT_VIEW = dict(eye=dict(x=0, y=0, z=2.2), up=dict(x=0, y=1, z=0))

def show_clouds(clouds, titles, height=380, color="curvature"):
    """Interactive 3D views of [N, 7] clouds (or [N, 3] raw points) side by side."""
    fig = make_subplots(rows=1, cols=len(clouds), specs=[[{"type": "scene"}] * len(clouds)],
                        subplot_titles=titles)
    for i, c in enumerate(clouds, start=1):
        if c.shape[1] >= 7 and color == "curvature":
            marker = dict(size=2, color=np.clip(c[:, 6], 0, None) ** CURVATURE_POWER, colorscale="Viridis")
        else:
            marker = dict(size=1.5, color=c[:, 2], colorscale="Greys")
        fig.add_trace(go.Scatter3d(x=c[:, 0], y=c[:, 1], z=c[:, 2], mode="markers", marker=marker), 1, i)
        fig.update_layout(**{"scene" + ("" if i == 1 else str(i)): dict(
            aspectmode="data", camera=FRONT_VIEW if c.shape[1] >= 7 else None,
            xaxis_visible=False, yaxis_visible=False, zaxis_visible=False)})
    fig.update_layout(height=height, showlegend=False, margin=dict(l=0, r=0, t=40, b=0))
    fig.show()

def gallery_probe(keys, info):
    """Per identity: earliest scan = gallery, later scans = probes. Returns index arrays into keys."""
    by_subject = defaultdict(list)
    for i, k in enumerate(keys):
        by_subject[info[k]["subject"]].append(i)
    gallery, probes = [], []
    for s, idx in sorted(by_subject.items()):
        idx = sorted(idx, key=lambda i: (info[keys[i]]["timestamp"], info[keys[i]]["image_id"]))
        gallery.append(idx[0])
        probes.extend(idx[1:])
    return np.array(gallery), np.array(probes)

def metrics_from_embeddings(emb, keys, info):
    """Gallery/probe metrics for L2-normalised embeddings [len(keys), D]."""
    g, p = gallery_probe(keys, info)
    labels = np.array([info[k]["subject"] for k in keys])
    return verification_metrics(emb[g], labels[g], emb[p], labels[p])

def metrics_from_similarity(sim_full, keys, info):
    """Gallery/probe metrics from a full [len(keys), len(keys)] similarity matrix."""
    g, p = gallery_probe(keys, info)
    labels = np.array([info[k]["subject"] for k in keys])
    return similarity_metrics(sim_full[np.ix_(g, p)], labels[g], labels[p])

def rank1_ci(sim_full, keys, info, n_boot=2000, seed=0):
    """95% bootstrap confidence interval of rank-1 (resampling probes)."""
    g, p = gallery_probe(keys, info)
    labels = np.array([info[k]["subject"] for k in keys])
    hits = labels[g][sim_full[np.ix_(g, p)].argmax(axis=0)] == labels[p]
    rng = np.random.RandomState(seed)
    boots = [hits[rng.randint(len(hits), size=len(hits))].mean() for _ in range(n_boot)]
    return np.percentile(boots, [2.5, 97.5])

## 3. Stage 1: index the dataset
A sample is a matching pair of a colour image and a points file with the same `Raw_{image_id}_{subject_id}_{timestamp}` prefix. Files without their partner are listed and ignored.

In [ ]:
pairs, unpaired = find_raw_pairs(RAW_DATA)
assert pairs, f"No Raw_*_color.png / Raw_*_points.npy pairs found under {RAW_DATA}"
INFO = {p["key"]: p for p in pairs}              # key -> {subject, timestamp, image_id, color, points}
per_subject = pd.Series([p["subject"] for p in pairs]).value_counts()

print(f"{len(pairs)} colour/depth pairs from {len(per_subject)} identities "
      f"({len(unpaired)} files without a partner ignored)")
print(f"samples per identity: min {per_subject.min()}, median {per_subject.median():.0f}, max {per_subject.max()}")
display(pd.DataFrame(pairs)[["subject", "image_id", "timestamp"]].head())
per_subject.plot.hist(bins=30, figsize=(8, 2.5), title="samples per identity"); plt.show()

### 1.1 Inspect one raw sample
Check that the layout below is what you expect. The preprocessing supports `[H, W, 3+]`, `[H·W, 3+]` (aligned with the colour image) and `[N, 3+]` arrays, in metres or millimetres. **Organized** clouds can be cropped with the face box from the colour image. Unorganized ones are cropped around the point closest to the camera.

In [ ]:
example = pairs[0]
image = Image.open(example["color"]).convert("RGB")
raw = np.load(example["points"], allow_pickle=False)
xyz, pixels, grid = load_raw_points(example["points"], image.size)
layout = ("organized grid" if raw.ndim == 3 else
          "flat, organized like the colour image" if grid is not None else "unorganized")
print(f"colour image  : {image.size[0]} x {image.size[1]}")
print(f"points array  : shape {raw.shape}, dtype {raw.dtype} -> {layout}")
print(f"valid points  : {len(xyz)} of {raw.reshape(-1, raw.shape[-1]).shape[0]}")
print("xyz range     :", np.round(xyz.min(0), 3), "to", np.round(xyz.max(0), 3),
      "(units look like", "millimetres)" if np.median(np.linalg.norm(xyz, axis=1)) > 20 else "metres)")

plt.figure(figsize=(4, 4)); plt.imshow(image); plt.axis("off"); plt.title(example["key"], fontsize=8); plt.show()
sub = xyz[np.random.RandomState(0).choice(len(xyz), min(20000, len(xyz)), replace=False)]
show_clouds([sub], ["raw points (camera coordinates)"], height=420)

## 4. Stage 2: face detection in the colour images
MTCNN finds the face in every colour image. When several faces are found, the largest confident one is kept. Samples whose best face has confidence below `PREPROCESS["min_face_prob"]` are dropped: a wrong or missing face would teach the network wrong colour features. The detections are cached in `detections.json`.

In [ ]:
DET_PATH = work("detections.json")
if os.path.isfile(DET_PATH):
    detections = load_json(DET_PATH)
    print(f"[skip] using cached {DET_PATH}")
else:
    mtcnn = MTCNN(image_size=160, margin=20, keep_all=True, device=DEVICE)
    detections = {}
    for p in tqdm(pairs, desc="detecting faces"):
        img = Image.open(p["color"]).convert("RGB")
        boxes, probs = mtcnn.detect(img)
        record = dict(image_size=list(img.size), box=None, prob=0.0, n_faces=0)
        if boxes is not None:
            confident = [i for i in range(len(boxes)) if probs[i] >= PREPROCESS["min_face_prob"]] or [int(np.argmax(probs))]
            best = max(confident, key=lambda i: (boxes[i][2] - boxes[i][0]) * (boxes[i][3] - boxes[i][1]))
            record.update(box=[float(v) for v in boxes[best]], prob=float(probs[best]), n_faces=len(boxes))
        detections[p["key"]] = record
    save_json(detections, DET_PATH)
    del mtcnn

face_ok = {k: d["box"] is not None and d["prob"] >= PREPROCESS["min_face_prob"] for k, d in detections.items()}
probs = np.array([d["prob"] for d in detections.values()])
print(f"face found with confidence >= {PREPROCESS['min_face_prob']}: {sum(face_ok.values())} / {len(face_ok)}")
print(f"no face at all: {(probs == 0).sum()} | low confidence: {((probs > 0) & (probs < PREPROCESS['min_face_prob'])).sum()} | "
      f"several faces: {sum(d['n_faces'] > 1 for d in detections.values())}")

In [ ]:
# Detections on a few samples (green = kept, red = rejected)
shown = [p for p in pairs if not face_ok[p["key"]]][:2] + [p for p in pairs if face_ok[p["key"]]][:6]
fig, axes = plt.subplots(1, len(shown), figsize=(2.6 * len(shown), 3))
for ax, p in zip(np.atleast_1d(axes), shown):
    d = detections[p["key"]]
    ax.imshow(Image.open(p["color"]).convert("RGB")); ax.axis("off")
    ax.set_title(f"{p['subject']}  p={d['prob']:.2f}", fontsize=8)
    if d["box"]:
        x1, y1, x2, y2 = d["box"]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=2,
                                       color="lime" if face_ok[p["key"]] else "red"))
plt.tight_layout(); plt.show()

## 5. Stage 3: preprocess the raw point clouds
Each raw cloud is converted into the same `[N, 7]` format the depth model was trained on:
1. **crop**: for organized clouds, the points inside the colour face box (enlarged by 15%), then everything within `face_radius_mm` of the nose tip (the point closest to the camera);
2. remove isolated outlier points;
3. compute **normals**, pointing towards the camera, and **curvature** (PCL's surface variation);
4. **rotate** from camera axes into the training frame (nose towards +z, y up);
5. **centre, scale into the unit sphere** and sample `NUM_POINTS` points.

Results are cached per sample in `clouds/<key>.npy`, so an interrupted run continues where it stopped. Samples that can't be converted are listed with the reason. Afterwards, the same quality audit as in `face_pipeline.ipynb` removes broken and outlier clouds.

In [ ]:
CLOUD_DIR = work("clouds")
FAIL_PATH = work("cloud_failures.json")
os.makedirs(CLOUD_DIR, exist_ok=True)
cloud_failures = load_json(FAIL_PATH) if os.path.isfile(FAIL_PATH) else {}

def preprocess_one(key):
    """Convert one raw scan; returns (key, error message or None)."""
    p, d = INFO[key], detections[key]
    box = d["box"] if PREPROCESS["use_face_box"] and face_ok[key] else None
    try:
        cloud = raw_to_cloud(p["points"], image_size=tuple(d["image_size"]), box=box, num_points=NUM_POINTS,
                             face_radius_mm=PREPROCESS["face_radius_mm"],
                             curvature_k=PREPROCESS["curvature_k"], camera=PREPROCESS["camera"])
    except (ValueError, OSError) as e:
        return key, str(e)
    np.save(os.path.join(CLOUD_DIR, key + ".npy"), cloud)
    return key, None

todo = [p["key"] for p in pairs if face_ok[p["key"]] and p["key"] not in cloud_failures
        and not os.path.isfile(os.path.join(CLOUD_DIR, p["key"] + ".npy"))]
if todo:
    with Pool(NUM_WORKERS) as pool:
        for key, error in tqdm(pool.imap_unordered(preprocess_one, todo, chunksize=4), total=len(todo),
                               desc="preprocessing clouds"):
            if error:
                cloud_failures[key] = error
    save_json(cloud_failures, FAIL_PATH)
else:
    print("[skip] every cloud is already preprocessed")

cloud_path = {k: os.path.join(CLOUD_DIR, k + ".npy") for k in INFO
              if os.path.isfile(os.path.join(CLOUD_DIR, k + ".npy"))}
print(f"{len(cloud_path)} clouds ready, {len(cloud_failures)} failed")
if cloud_failures:
    display(pd.Series(list(cloud_failures.values())).str.replace(r"\d+", "N", regex=True)
            .value_counts().rename("samples").to_frame())

In [ ]:
# Quality audit of the processed clouds (cached)
AUDIT_PATH = work("cloud_audit.json")
audit_keys = sorted(cloud_path)
if os.path.isfile(AUDIT_PATH) and load_json(AUDIT_PATH)["keys"] == audit_keys:
    cloud_stats = load_json(AUDIT_PATH)["stats"]
    print(f"[skip] using cached {AUDIT_PATH}")
else:
    cloud_stats = audit_scans([cloud_path[k] for k in audit_keys], workers=NUM_WORKERS)
    save_json({"keys": audit_keys, "stats": cloud_stats}, AUDIT_PATH)
reasons = flag_scans(cloud_stats, {"min_points": NUM_POINTS // 2})
cloud_ok = {k: not r for k, r in zip(audit_keys, reasons)}
print(f"{sum(cloud_ok.values())} of {len(audit_keys)} clouds pass the audit")
if any(reasons):
    display(pd.Series([x for r in reasons for x in r]).value_counts().rename("clouds").to_frame())

# Orientation check: after rotation into the face frame the normals should point towards +z
mean_nz = np.array([s.get("mean_normal_z", np.nan) for s in cloud_stats])
print(f"clouds whose mean normal points towards +z: {np.nanmean(mean_nz > 0):.1%} "
      "(should be close to 100%; if it is near 0%, switch PREPROCESS['camera'])")

In [ ]:
# Compare with the data the depth model was trained on (optional): the statistics should be similar
if REFERENCE_DATA:
    ref_files = find_npy_files(REFERENCE_DATA)
    ref_files = [ref_files[i] for i in np.random.RandomState(SEED).choice(len(ref_files), min(300, len(ref_files)), replace=False)]
    ref_stats = audit_scans(ref_files, workers=NUM_WORKERS, progress=False)
    cols = ["num_points", "max_radius", "pca_mid_ratio", "pca_flat_ratio", "nn_dist_median",
            "mean_normal_z", "curv_max"]
    display(pd.DataFrame({"new data (median)": pd.DataFrame(cloud_stats)[cols].median(),
                          "reference (median)": pd.DataFrame(ref_stats)[cols].median()}).round(4))
    show_clouds([np.load(cloud_path[audit_keys[0]]), np.load(ref_files[0])],
                ["new: " + audit_keys[0], "reference: " + os.path.basename(ref_files[0])])

In [ ]:
# A few samples: colour image, then the processed cloud the depth model sees
for key in [k for k in audit_keys if cloud_ok[k]][:3]:
    fig = plt.figure(figsize=(3, 3)); plt.imshow(Image.open(INFO[key]["color"]).convert("RGB"))
    x1, y1, x2, y2 = detections[key]["box"]
    plt.gca().add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="lime", lw=2))
    plt.axis("off"); plt.title(key, fontsize=7); plt.show()
    cloud = np.load(cloud_path[key])
    show_clouds([cloud, cloud], ["curvature", "depth"], height=330, color="curvature")

## 6. Stage 4: identity-disjoint train / val / test split
Only **usable** samples are kept: a confident face in the colour image, and a processed cloud that passed the audit. Identities with fewer than `SPLIT["min_samples"]` usable samples are dropped. The remaining identities are shuffled once (`SEED`) and divided into train / val / test, so no person appears in two splits. The split is saved, so it never changes between runs.

In [ ]:
SPLIT_PATH = work("splits.json")
if os.path.isfile(SPLIT_PATH):
    splits = load_json(SPLIT_PATH)
    print(f"[skip] using cached {SPLIT_PATH}")
else:
    usable = [k for k in INFO if face_ok.get(k) and cloud_ok.get(k)]
    by_subject = defaultdict(list)
    for k in usable:
        by_subject[INFO[k]["subject"]].append(k)
    subjects = sorted(s for s, ks in by_subject.items() if len(ks) >= SPLIT["min_samples"])
    np.random.RandomState(SEED).shuffle(subjects)
    n_test = max(1, round(len(subjects) * SPLIT["test"]))
    n_val = max(1, round(len(subjects) * SPLIT["val"]))
    parts = {"test": subjects[:n_test], "val": subjects[n_test:n_test + n_val], "train": subjects[n_test + n_val:]}
    splits = {name: sorted(k for s in subs for k in by_subject[s]) for name, subs in parts.items()}
    save_json(splits, SPLIT_PATH)

ids = {name: {INFO[k]["subject"] for k in keys} for name, keys in splits.items()}
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"])
rows = {}
for name in ("train", "val", "test"):
    row = dict(identities=len(ids[name]), samples=len(splits[name]))
    if name != "train":
        g, p = gallery_probe(splits[name], INFO)
        row["gallery / probes"] = f"{len(g)} / {len(p)}"
    rows[name] = row
display(pd.DataFrame(rows).T.fillna(""))

## 7. Stage 5: frozen embeddings
Both backbones stay frozen, so their embeddings are computed once and cached. The fusion network then trains on the cached vectors, which takes seconds per epoch.
* **val / test**: one clean embedding per sample and modality.
* **train**: `TRAIN_VIEWS` embeddings per sample and modality: one clean, the rest from **augmented inputs**. Colour: horizontal flip, small rotation / shift / zoom, brightness and contrast changes, a random erased patch. Depth: rotation, jitter, and cut-out holes. Every training step draws a random view per modality, independently, so the network rarely sees the same colour/depth combination twice.

In [ ]:
def augment_face(face):
    """Colour augmentation of an aligned, standardised face tensor [3, 160, 160] (values about -1..1)."""
    if np.random.rand() < 0.5:
        face = face.flip(-1)
    angle = np.random.uniform(-RGB_AUG["max_rotation_deg"], RGB_AUG["max_rotation_deg"])
    shift = [int(np.random.uniform(-1, 1) * RGB_AUG["max_shift"] * 160) for _ in range(2)]
    face = TF.affine(face, angle=angle, translate=shift, scale=np.random.uniform(*RGB_AUG["scale"]),
                     shear=[0.0], fill=-1.0)
    face = face + np.random.uniform(-RGB_AUG["brightness"], RGB_AUG["brightness"])
    mean = face.mean()
    face = (face - mean) * np.random.uniform(1 - RGB_AUG["contrast"], 1 + RGB_AUG["contrast"]) + mean
    if np.random.rand() < RGB_AUG["erase_prob"]:
        h, w = np.random.randint(20, 60, size=2)
        y, x = np.random.randint(0, 160 - h), np.random.randint(0, 160 - w)
        face = face.clone(); face[:, y:y + h, x:x + w] = 0.0
    return face.clamp(-1.0, 1.0)

def augment_cloud(points):
    """Depth augmentation of a [N, 7] tensor: rotation (points and normals), jitter, a cut-out hole."""
    a = np.radians(np.random.uniform(-DEPTH_AUG["max_rotation_deg"], DEPTH_AUG["max_rotation_deg"], 3))
    rot = (data_utils.angle_axis(a[2], np.array([0.0, 0.0, 1.0])) @ data_utils.angle_axis(a[1], np.array([0.0, 1.0, 0.0]))
           @ data_utils.angle_axis(a[0], np.array([1.0, 0.0, 0.0])))
    points[:, 0:3] = points[:, 0:3] @ rot.t()
    points[:, 3:6] = points[:, 3:6] @ rot.t()
    points[:, 0:3] += torch.randn(len(points), 3) * DEPTH_AUG["jitter_std"]
    if np.random.rand() < DEPTH_AUG["cutout_prob"]:
        n = len(points)
        n_cut = int(n * np.random.uniform(0.02, DEPTH_AUG["cutout_max_frac"]))
        keep = torch.argsort((points[:, 0:3] - points[np.random.randint(n), 0:3]).norm(dim=1))[n_cut:]
        points = points[torch.cat([keep, keep[torch.randint(len(keep), (n_cut,))]])]
    return points

In [ ]:
EMB_PATH = work("embeddings.npz")

@torch.no_grad()
def embed_split(keys, views, rgb_net, depth_net, mtcnn):
    """Embeddings [len(keys), views, 512] for both modalities; view 0 is always clean."""
    rgb = np.zeros((len(keys), views, 512), np.float32)
    depth = np.zeros((len(keys), views, 512), np.float32)
    for start in tqdm(range(0, len(keys), EVAL_BATCH), desc=f"embedding ({views} view(s))"):
        batch = keys[start:start + EVAL_BATCH]
        faces = [mtcnn.extract(Image.open(INFO[k]["color"]).convert("RGB"),          # aligned [3, 160, 160]
                               np.array([detections[k]["box"]], dtype=np.float32), None) for k in batch]
        clouds = [load_npy_cloud(cloud_path[k], NUM_POINTS, CURVATURE_POWER) for k in batch]
        for v in range(views):
            f = torch.stack([augment_face(x) if v else x for x in faces]).to(DEVICE)
            c = torch.stack([augment_cloud(x.clone()) if v else x for x in clouds]).to(DEVICE)
            rgb[start:start + len(batch), v] = F.normalize(rgb_net(f), dim=1).cpu().numpy()
            depth[start:start + len(batch), v] = F.normalize(depth_net(c), dim=1).cpu().numpy()
    return rgb, depth

if os.path.isfile(EMB_PATH):
    EMB = dict(np.load(EMB_PATH))
    print(f"[skip] using cached {EMB_PATH}")
else:
    torch.manual_seed(SEED); np.random.seed(SEED)
    mtcnn = MTCNN(image_size=160, margin=20, post_process=True, device=DEVICE)
    rgb_net = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)
    depth_net = Pointnet2FaceClsSSG(input_channels=3, use_xyz=True).to(DEVICE)
    ckpt = torch.load(PK_CKPT, map_location="cpu", weights_only=False)
    depth_net.load_state_dict(ckpt.get("model_state", ckpt))
    depth_net.eval()
    EMB = {}
    for name in ("train", "val", "test"):
        EMB[f"{name}_rgb"], EMB[f"{name}_depth"] = embed_split(
            splits[name], TRAIN_VIEWS if name == "train" else 1, rgb_net, depth_net, mtcnn)
    np.savez(EMB_PATH, **EMB)
    del mtcnn, rgb_net, depth_net
    torch.cuda.empty_cache()
print({k: v.shape for k, v in EMB.items()})

## 8. Stage 6: baselines
Before training anything, how good is each frozen model alone, and how far does **score-level fusion** get? Score-level fusion is simply `w · cos_rgb + (1 − w) · cos_depth`, with `w` tuned on val. The learned fusion network has to beat this to be worth it.

In [ ]:
def sims(split):
    """Cosine-similarity matrices of the clean embeddings: (rgb, depth)."""
    r, d = EMB[f"{split}_rgb"][:, 0], EMB[f"{split}_depth"][:, 0]
    return r @ r.T, d @ d.T

val_sim_rgb, val_sim_depth = sims("val")
weights = np.round(np.arange(0, 1.0001, 0.05), 2)
val_curve = [metrics_from_similarity(w * val_sim_rgb + (1 - w) * val_sim_depth, splits["val"], INFO)[0][SELECT_METRIC]
             for w in weights]
SCORE_W = float(weights[int(np.argmax(val_curve))])

plt.figure(figsize=(6, 3))
plt.plot(weights, val_curve, marker="o"); plt.axvline(SCORE_W, color="gray", ls="--")
plt.xlabel("w (weight of colour)"); plt.ylabel(f"val {SELECT_METRIC}"); plt.title("Score-level fusion on val")
plt.grid(alpha=0.3); plt.show()

baseline_val = {"colour only (FaceNet)": metrics_from_similarity(val_sim_rgb, splits["val"], INFO)[0],
                "depth only (PK model)": metrics_from_similarity(val_sim_depth, splits["val"], INFO)[0],
                f"score fusion (w={SCORE_W})": metrics_from_similarity(
                    SCORE_W * val_sim_rgb + (1 - SCORE_W) * val_sim_depth, splits["val"], INFO)[0]}
display(pd.DataFrame(baseline_val).T.round(4))

## 9. Stage 7: fusion network
Both inputs are L2-normalised and pass through a **LayerNorm**, so neither modality wins simply by having larger numbers. Each is projected to 256-d. The two projections are concatenated, and an MLP produces the fused, L2-normalised 256-d embedding.

**Losses**, all with CosFace (a cosine classifier with margin `cosface_m`) over the training identities:
* `loss_fused` on the fused embedding; this is the main objective;
* `aux_weight × (loss_rgb + loss_depth)` on each projection alone (anti-laziness 2).

**Each training step:**
1. draw a random view per sample and per modality;
2. hide one modality for `modality_dropout` of the samples by replacing its projection with a learned "missing" vector (anti-laziness 1). The auxiliary loss of a hidden modality is skipped for those samples;
3. compute the losses and backpropagate;
4. apply **OGM**: compare the auxiliary classifiers' mean confidence in the true identity, `s_rgb` and `s_depth`. The dominant branch's gradients are multiplied by `1 − tanh(ogm_alpha · (ratio − 1))`, where ratio = dominant / weaker (anti-laziness 3);
5. take an AdamW step, with a cosine learning-rate schedule.

**Every epoch**, the val metrics are computed for the fused embedding, and also with each modality hidden. `reliance_rgb` is the drop in val TAR@FAR=1e-2 when the colour input is hidden; `reliance_depth` is the same for depth. If one reliance is near zero, the model ignores that modality. Early stopping keeps the best epoch by val `SELECT_METRIC`.

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, in_dim=512, proj_dim=256, hidden=512, dropout=0.3):
        super().__init__()
        def projection():
            return nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, hidden), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(hidden, proj_dim))
        self.proj_rgb, self.proj_depth = projection(), projection()
        self.missing_rgb = nn.Parameter(torch.zeros(proj_dim))     # stands in for a hidden modality
        self.missing_depth = nn.Parameter(torch.zeros(proj_dim))
        self.fuse = nn.Sequential(nn.Linear(2 * proj_dim, hidden), nn.GELU(), nn.Dropout(dropout),
                                  nn.Linear(hidden, proj_dim))

    def forward(self, rgb, depth, hide_rgb=None, hide_depth=None):
        z_rgb, z_depth = self.proj_rgb(rgb), self.proj_depth(depth)
        if hide_rgb is not None:
            z_rgb = torch.where(hide_rgb[:, None], self.missing_rgb.expand_as(z_rgb), z_rgb)
        if hide_depth is not None:
            z_depth = torch.where(hide_depth[:, None], self.missing_depth.expand_as(z_depth), z_depth)
        fused = F.normalize(self.fuse(torch.cat([z_rgb, z_depth], dim=1)), dim=1)
        return fused, z_rgb, z_depth

@torch.no_grad()
def fused_embeddings(model, split, hide=None):
    """Fused embeddings of a split's clean views; hide = None | 'rgb' | 'depth'."""
    model.eval()
    rgb = torch.from_numpy(EMB[f"{split}_rgb"][:, 0]).to(DEVICE)
    depth = torch.from_numpy(EMB[f"{split}_depth"][:, 0]).to(DEVICE)
    mask = torch.ones(len(rgb), dtype=torch.bool, device=DEVICE)
    fused, _, _ = model(rgb, depth, hide_rgb=mask if hide == "rgb" else None,
                        hide_depth=mask if hide == "depth" else None)
    return fused.cpu().numpy()

def cosine_probs(head, z, labels):
    """Mean probability of the true identity under a margin-free cosine classifier."""
    logits = head.s * F.linear(F.normalize(z), F.normalize(head.weight))
    return logits.softmax(dim=1).gather(1, labels[:, None]).mean()

In [ ]:
FUSION_DIR = work("fusion")
os.makedirs(FUSION_DIR, exist_ok=True)
FUSION_FILES = {k: os.path.join(FUSION_DIR, f) for k, f in
                dict(last="last.pth.tar", best="best.pth.tar", final="final.pth.tar", history="history.json").items()}

def train_fusion():
    # ---- training tensors (all on the GPU: the embeddings are small) ----
    train_keys = splits["train"]
    subjects = sorted({INFO[k]["subject"] for k in train_keys})
    labels = torch.tensor([subjects.index(INFO[k]["subject"]) for k in train_keys], device=DEVICE)
    R = torch.from_numpy(EMB["train_rgb"]).to(DEVICE)       # [N, views, 512]
    D = torch.from_numpy(EMB["train_depth"]).to(DEVICE)
    n, views = R.shape[:2]

    # ---- model, identity heads, optimiser ----
    model = FusionNet(512, FUSION["proj_dim"], FUSION["hidden"], FUSION["dropout"]).to(DEVICE)
    make_head = lambda: MarginCosineProduct(FUSION["proj_dim"], len(subjects),
                                            s=FUSION["cosface_s"], m=FUSION["cosface_m"]).to(DEVICE)
    head_fused, head_rgb, head_depth = make_head(), make_head(), make_head()
    params = [*model.parameters(), *head_fused.parameters(), *head_rgb.parameters(), *head_depth.parameters()]
    optimizer = torch.optim.AdamW(params, lr=FUSION["lr"], weight_decay=FUSION["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FUSION["epochs"])
    parts = dict(model=model, head_fused=head_fused, head_rgb=head_rgb, head_depth=head_depth,
                 optimizer=optimizer, scheduler=scheduler)

    start_epoch, history, best, stale = 0, [], -1.0, 0
    if os.path.isfile(FUSION_FILES["last"]):
        state = torch.load(FUSION_FILES["last"], map_location="cpu", weights_only=False)
        for name, part in parts.items():
            part.load_state_dict(state[name])
        start_epoch, history, best, stale = state["epoch"], state["history"], state["best"], state["stale"]
        print(f"Resuming after epoch {start_epoch}")
    print(f"{len(subjects)} training identities | {n} samples x {views} views | "
          f"{-(-n // FUSION['batch_size'])} steps per epoch")

    for epoch in range(start_epoch, FUSION["epochs"]):
        model.train()
        avg = {k: Mean() for k in ("loss", "fused", "rgb", "depth", "coef_rgb", "coef_depth")}
        for idx in torch.randperm(n, device=DEVICE).split(FUSION["batch_size"]):
            y = labels[idx]
            rgb = R[idx, torch.randint(views, (len(idx),), device=DEVICE)]      # random view per modality
            depth = D[idx, torch.randint(views, (len(idx),), device=DEVICE)]

            # 1) modality dropout: hide one modality (never both) for some samples
            hide = torch.rand(len(idx), device=DEVICE) < ANTI_LAZY["modality_dropout"]
            hide_rgb_first = torch.rand(len(idx), device=DEVICE) < 0.5
            hide_rgb, hide_depth = hide & hide_rgb_first, hide & ~hide_rgb_first

            fused, z_rgb, z_depth = model(rgb, depth, hide_rgb, hide_depth)

            # 2) main loss + auxiliary per-modality losses (only where that modality is visible)
            loss_fused = F.cross_entropy(head_fused(fused, y), y)
            ce_rgb = F.cross_entropy(head_rgb(z_rgb, y), y, reduction="none")
            ce_depth = F.cross_entropy(head_depth(z_depth, y), y, reduction="none")
            loss_rgb = ce_rgb[~hide_rgb].mean()
            loss_depth = ce_depth[~hide_depth].mean()
            loss = loss_fused + ANTI_LAZY["aux_weight"] * (loss_rgb + loss_depth)

            optimizer.zero_grad()
            loss.backward()

            # 3) OGM: slow down the branch that is currently ahead
            coef_rgb = coef_depth = 1.0
            if ANTI_LAZY["ogm_alpha"] > 0:
                with torch.no_grad():
                    s_rgb = cosine_probs(head_rgb, z_rgb[~hide_rgb], y[~hide_rgb])
                    s_depth = cosine_probs(head_depth, z_depth[~hide_depth], y[~hide_depth])
                    ratio = (s_rgb / s_depth.clamp_min(1e-8)).item()
                if ratio > 1:
                    coef_rgb = 1 - np.tanh(ANTI_LAZY["ogm_alpha"] * (ratio - 1))
                else:
                    coef_depth = 1 - np.tanh(ANTI_LAZY["ogm_alpha"] * (1 / max(ratio, 1e-8) - 1))
                for p in model.proj_rgb.parameters():
                    p.grad.mul_(coef_rgb)
                for p in model.proj_depth.parameters():
                    p.grad.mul_(coef_depth)
            optimizer.step()

            for k, v in dict(loss=loss.item(), fused=loss_fused.item(), rgb=loss_rgb.item(),
                             depth=loss_depth.item(), coef_rgb=coef_rgb, coef_depth=coef_depth).items():
                avg[k].add(v, len(idx))
        lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        # ---- validation: fused, and with each modality hidden ----
        val, _ = metrics_from_embeddings(fused_embeddings(model, "val"), splits["val"], INFO)
        no_rgb, _ = metrics_from_embeddings(fused_embeddings(model, "val", hide="rgb"), splits["val"], INFO)
        no_depth, _ = metrics_from_embeddings(fused_embeddings(model, "val", hide="depth"), splits["val"], INFO)
        row = dict(epoch=epoch + 1, lr=lr, **{f"train_{k}": m.value for k, m in avg.items()}, **val,
                   val_depth_only=no_rgb["tar@far=0.01"], val_rgb_only=no_depth["tar@far=0.01"],
                   reliance_rgb=val["tar@far=0.01"] - no_rgb["tar@far=0.01"],
                   reliance_depth=val["tar@far=0.01"] - no_depth["tar@far=0.01"])
        history.append(row)
        improved = row[SELECT_METRIC] > best
        if improved:
            best, stale = row[SELECT_METRIC], 0
            torch.save(dict(model=model.state_dict(), epoch=epoch + 1, metrics=row, subjects=subjects,
                            config=dict(FUSION=FUSION, ANTI_LAZY=ANTI_LAZY)), FUSION_FILES["best"])
        else:
            stale += 1
        if epoch % 5 == 0 or improved:
            print(f"epoch {epoch + 1:3d} | loss {row['train_loss']:.3f} (fused {row['train_fused']:.3f}, "
                  f"rgb {row['train_rgb']:.3f}, depth {row['train_depth']:.3f}) | OGM rgb {row['train_coef_rgb']:.2f} "
                  f"depth {row['train_coef_depth']:.2f} | val rank1 {val['rank1']:.3f} tar@1e-2 {val['tar@far=0.01']:.3f} "
                  f"| reliance rgb {row['reliance_rgb']:+.3f} depth {row['reliance_depth']:+.3f}"
                  + ("  <- best" if improved else ""))
        torch.save(dict(epoch=epoch + 1, history=history, best=best, stale=stale,
                        **{name: part.state_dict() for name, part in parts.items()}), FUSION_FILES["last"])
        save_json(history, FUSION_FILES["history"])
        if stale >= FUSION["patience"]:
            print(f"Early stopping: no better val {SELECT_METRIC} for {stale} epochs")
            break
    torch.save(dict(model=model.state_dict(), epoch=len(history)), FUSION_FILES["final"])

In [ ]:
if os.path.isfile(FUSION_FILES["final"]):
    print(f"[skip] fusion training already finished ({FUSION_FILES['final']})")
else:
    train_fusion()

history = load_json(FUSION_FILES["history"])
h = pd.DataFrame(history)
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
h.plot(x="epoch", y=["train_fused", "train_rgb", "train_depth"], ax=axes[0], title="training losses")
h.plot(x="epoch", y=["rank1", "tar@far=0.01", "tar@far=0.001"], ax=axes[1], title="val (fused)")
h.plot(x="epoch", y=["tar@far=0.01", "val_rgb_only", "val_depth_only"], ax=axes[2],
       title="val TAR@1e-2: fused vs one modality")
h.plot(x="epoch", y=["reliance_rgb", "reliance_depth"], ax=axes[3], title="reliance (drop when hidden)")
for ax in axes:
    ax.grid(alpha=0.3)
best_epoch = load_json(FUSION_FILES["history"])[int(h[SELECT_METRIC].idxmax())]["epoch"]
for ax in axes:
    ax.axvline(best_epoch, color="gray", ls="--", lw=1)
plt.tight_layout(); plt.show()

## 10. Stage 8: final evaluation on the test set
The test identities were never used: not for training, not for early stopping, and not for tuning the score-fusion weight. Every method is scored on exactly the same gallery and probes:

| method | what it is |
|---|---|
| colour only | FaceNet embeddings, cosine similarity |
| depth only | PK-model embeddings, cosine similarity |
| score fusion | `w · cos_rgb + (1 − w) · cos_depth`, with `w` from val |
| **learned fusion** | the fusion network (best val epoch) |
| learned fusion, colour hidden | the same network with only the depth input |
| learned fusion, depth hidden | the same network with only the colour input |

`rank1 95% CI` is a bootstrap interval over probes. With a small test set, differences inside these intervals aren't meaningful.

In [ ]:
best_fusion = FusionNet(512, FUSION["proj_dim"], FUSION["hidden"], FUSION["dropout"]).to(DEVICE)
best_fusion.load_state_dict(torch.load(FUSION_FILES["best"], map_location="cpu", weights_only=False)["model"])

test_keys = splits["test"]
test_sim_rgb, test_sim_depth = sims("test")
fused_test = fused_embeddings(best_fusion, "test")
methods = {
    "colour only (FaceNet)": test_sim_rgb,
    "depth only (PK model)": test_sim_depth,
    f"score fusion (w={SCORE_W})": SCORE_W * test_sim_rgb + (1 - SCORE_W) * test_sim_depth,
    "learned fusion": fused_test @ fused_test.T,
    "learned fusion, colour hidden": (lambda e: e @ e.T)(fused_embeddings(best_fusion, "test", hide="rgb")),
    "learned fusion, depth hidden": (lambda e: e @ e.T)(fused_embeddings(best_fusion, "test", hide="depth")),
}
results, curves = {}, {}
for name, sim in methods.items():
    results[name], curves[name] = metrics_from_similarity(sim, test_keys, INFO)
    lo, hi = rank1_ci(sim, test_keys, INFO)
    results[name]["rank1 95% CI"] = f"{lo:.3f} - {hi:.3f}"
table = pd.DataFrame(results).T
table.to_csv(work("results_test.csv"))
g, p = gallery_probe(test_keys, INFO)
print(f"Test: {len(ids['test'])} identities, {len(g)} gallery samples, {len(p)} probes")
display(table)

In [ ]:
# ROC and CMC on the test set
fig, (ax_roc, ax_cmc) = plt.subplots(1, 2, figsize=(15, 5))
for name, c in curves.items():
    style = "-" if "hidden" not in name else ":"
    ax_roc.semilogx(np.maximum(c["fpr"], 1e-5), c["tpr"], style, label=f"{name} (AUC {results[name]['auc']:.3f})")
    ax_cmc.plot(np.arange(1, len(c["cmc"]) + 1), c["cmc"], style, marker="o", label=name)
ax_roc.axvline(1e-2, color="gray", ls="--", lw=1)
ax_roc.set(title="ROC (test)", xlabel="false accept rate (log)", ylabel="true accept rate", xlim=(1e-4, 1))
ax_cmc.set(title="CMC (test)", xlabel="rank", ylabel="identification rate")
for ax in (ax_roc, ax_cmc):
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Genuine vs impostor score distributions: each modality alone vs the learned fusion
fig, axes = plt.subplots(1, 3, figsize=(18, 3.5))
for ax, name in zip(axes, ["colour only (FaceNet)", "depth only (PK model)", "learned fusion"]):
    ax.hist(curves[name]["impostor"], bins=80, density=True, alpha=0.6, label="impostor")
    ax.hist(curves[name]["genuine"], bins=80, density=True, alpha=0.6, label="genuine")
    ax.set_title(f"{name}  (EER {results[name]['eer']:.3f})"); ax.legend()
plt.tight_layout(); plt.show()

# How much each modality contributes to the learned fusion on test
drop = {"hide colour": results["learned fusion"]["tar@far=0.01"] - results["learned fusion, colour hidden"]["tar@far=0.01"],
        "hide depth": results["learned fusion"]["tar@far=0.01"] - results["learned fusion, depth hidden"]["tar@far=0.01"]}
plt.figure(figsize=(5, 3)); plt.bar(list(drop), list(drop.values()), color=["tab:orange", "tab:blue"])
plt.ylabel("drop in test TAR@FAR=1e-2"); plt.title("Modality reliance (both should be clearly > 0)")
plt.grid(alpha=0.3, axis="y"); plt.show()

In [ ]:
# t-SNE of the fused test embeddings (one colour per identity, first 15 identities)
shown_ids = sorted(ids["test"])[:15]
sel = [i for i, k in enumerate(test_keys) if INFO[k]["subject"] in shown_ids]
xy = TSNE(n_components=2, perplexity=min(30, len(sel) - 1), init="pca", random_state=SEED).fit_transform(fused_test[sel])
plt.figure(figsize=(7, 6))
plt.scatter(xy[:, 0], xy[:, 1], c=[shown_ids.index(INFO[test_keys[i]]["subject"]) for i in sel], cmap="tab20", s=30)
plt.title("t-SNE of fused test embeddings"); plt.xticks([]); plt.yticks([]); plt.show()

In [ ]:
# Failure cases of the learned fusion: probe | its true gallery sample | the wrongly matched gallery sample
sim = methods["learned fusion"][np.ix_(g, p)]
labels = np.array([INFO[k]["subject"] for k in test_keys])
pred = sim.argmax(axis=0)
wrong = np.flatnonzero(labels[g][pred] != labels[p])
print(f"{len(wrong)} of {len(p)} test probes misidentified by the learned fusion")
for j in wrong[:3]:
    probe = test_keys[p[j]]
    true_g = test_keys[g[list(labels[g]).index(labels[p[j]])]]
    wrong_g = test_keys[g[pred[j]]]
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    for ax, key, title in zip(axes, [probe, true_g, wrong_g], ["probe", "true identity", "matched (wrong)"]):
        ax.imshow(Image.open(INFO[key]["color"]).convert("RGB")); ax.axis("off")
        ax.set_title(f"{title}\n{INFO[key]['subject']}", fontsize=9)
    plt.tight_layout(); plt.show()
    show_clouds([np.load(cloud_path[k]) for k in (probe, true_g, wrong_g)], ["probe", "true identity", "matched (wrong)"],
                height=300)

## Reading the results
* **Learned fusion should beat both single modalities and score fusion.** If it only matches score fusion, the simple weighted sum is the better choice: it has nothing to train and fewer ways to overfit.
* **Both reliance bars should be clearly above zero.** If hiding one modality costs almost nothing, the network ignores it. Then raise `ANTI_LAZY["modality_dropout"]` or `ogm_alpha`, or check that modality's data quality (sections 4 and 5).
* **"Colour hidden" vs "depth only".** If the fusion network with colour hidden scores far below the plain depth embedding, the depth projection was neglected in training. Raise `aux_weight`.
* **Small test sets are noisy**: compare methods by the rank-1 confidence intervals.

All results are in `runs/fusion/`. To retrain the fusion network, delete `runs/fusion/fusion/`. To change preprocessing or augmentation, also delete `embeddings.npz`, plus `clouds/` and `cloud_audit.json` if the preprocessing changed.